In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('dump_data.csv')
print(df)
print(df.isna().sum())
df_clean = df.dropna()

    Index      Customer Id First Name Last Name  \
0       1  DD37Cf93aecA6Dc     Sheryl    Baxter   
1       2  1Ef7b82A4CAAD10    Preston    Lozano   
2       3  6F94879bDAfE5a6        Roy     Berry   
3       4  5Cef8BFA16c5e3c      Linda     Olsen   
4       5  053d585Ab6b3159     Joanna    Bender   
..    ...              ...        ...       ...   
95     96  cb8E23e48d22Eae       Karl     Greer   
96     97  CeD220bdAaCfaDf       Lynn  Atkinson   
97     98  28CDbC0dFe4b1Db       Fred    Guerra   
98     99  c23d1D9EE8DEB0A     Yvonne    Farmer   
99    100  2354a0E336A91A1   Clarence    Haynes   

                            Company               City  \
0                   Rasmussen Group       East Leonard   
1                       Vega-Gentry  East Jimmychester   
2                     Murillo-Perry      Isabelborough   
3   Dominguez, Mcmillan and Donovan         Bensonview   
4          Martin, Lang and Andrade     West Priscilla   
..                              ...    

In [2]:
# Điền số 0 vào cột số
df['Phone 1'] = df['Phone 1'].fillna(0)

# Điền "Unknown" vào cột chữ
df['Company'] = df['Company'].fillna('Unknown')

# # Điền giá trị trung bình (rất hay dùng)
# df['Age'] = df['Age'].fillna(df['Age'].mean())

# Điền giá trị trước/sau (forward/backward fill)
df['Subscription Date'] = df['Subscription Date'].ffill()

In [3]:
# Làm sạch tất cả cột chữ một lần
text_cols = ['First Name', 'Last Name', 'Company', 'City', 'Country']

for col in text_cols:
    if col in df.columns:
        df[col] = df[col].astype(str)           # đảm bảo là string
        df[col] = df[col].str.strip()           # bỏ khoảng trắng đầu cuối
        df[col] = df[col].str.lower()           # chuyển hết về chữ thường
        df[col] = df[col].str.replace(r'\s+', ' ', regex=True)  # bỏ khoảng trắng thừa

In [4]:
df['First Name'] = df['First Name'].str.strip().str.lower()
print(df['First Name'])

0       sheryl
1      preston
2          roy
3        linda
4       joanna
        ...   
95        karl
96        lynn
97        fred
98      yvonne
99    clarence
Name: First Name, Length: 100, dtype: str


In [6]:
# Cột có thể bẩn: "1,200", "1 200", "N/A", "", "$500", "500.00 VND"...

# Cách mạnh nhất và dễ nhất:
df['Customer Id'] = pd.to_numeric(df['Customer Id'], errors='coerce')
# → "N/A", "" sẽ thành NaN tự động

# Nếu có dấu phẩy (thousands separator)
# df['Salary'] = df['Salary'].str.replace(',', '', regex=False)   # bỏ dấu phẩy
# df['Salary'] = pd.to_numeric(df['Salary'], errors='coerce')

# Hoặc dùng một hàm sạch luôn:
def clean_number(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    x = x.replace(',', '').replace('$', '').replace('VND', '')
    try:
        return float(x)
    except:
        return np.nan

# df['Some_Column'] = df['Some_Column'].apply(clean_number)

In [7]:
import pandas as pd
import numpy as np
import os

In [8]:
os.makedirs("data", exist_ok=True)

dirty = pd.DataFrame({
    "order_id": [1, 2, 3, 4, 5, 6, 7],
    "customer": [" an ", "Binh", "AN", "chi ", None, " dung", "lan"],
    "amount": ["200", "1,250", "N/A", "", "300", " 150 ", "90"],
    "region": [" North ", "south", "NORTH", None, "South ", " south", "north"],
})

csv_path = "data/orders_dirty.csv"
dirty.to_csv(csv_path, index=False)
print(csv_path)
dirty

data/orders_dirty.csv


,order_id,customer,amount,region
0,1,an,200,North
1,2,Binh,"1,250",south
2,3,AN,N/A,NORTH
3,4,chi,,NaN
4,5,NaN,300,South
5,6,dung,150,south
6,7,lan,90,north


In [9]:
df = pd.read_csv(csv_path)

print(df.head())
print()
print("shape:", df.shape)
print()
df.info()
print()
print("Missing by column:\n", df.isna().sum())

   order_id customer amount   region
0         1      an     200   North 
1         2     Binh  1,250    south
2         3       AN    NaN    NORTH
3         4     chi     NaN      NaN
4         5      NaN    300   South 

shape: (7, 4)

<class 'pandas.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   order_id  7 non-null      int64
 1   customer  6 non-null      str  
 2   amount    5 non-null      str  
 3   region    6 non-null      str  
dtypes: int64(1), str(3)
memory usage: 356.0 bytes

Missing by column:
 order_id    0
customer    1
amount      2
region      1
dtype: int64


In [10]:
df_clean = df.copy()

text_cols = ["customer", "region"]

for col in text_cols:
    df_clean[col] = (
        df_clean[col]
        .astype("string")   # an toàn hơn khi có missing
        .str.strip()
        .str.lower()
    )

print(df_clean[["customer", "region"]].head(7))
print()
print("Unique regions:", df_clean["region"].dropna().unique())

  customer region
0       an  north
1     binh  south
2       an  north
3      chi   <NA>
4     <NA>  south
5     dung  south
6      lan  north

Unique regions: <StringArray>
['north', 'south']
Length: 2, dtype: string


In [11]:
df_clean["amount"] = (
    df_clean["amount"]
    .astype("string")
    .str.strip()
    .str.replace(",", "", regex=False)
)

df_clean["amount"] = pd.to_numeric(df_clean["amount"], errors="coerce")

print("After parse dtype:", df_clean["amount"].dtype)
print(df_clean["amount"])

After parse dtype: Int64
0     200
1    1250
2    <NA>
3    <NA>
4     300
5     150
6      90
Name: amount, dtype: Int64
